# Projeto 02 - Cenário 3D: Casa Minecraft

Cenário 3D renderizado com **OpenGL** (PyOpenGL) e **GLFW**, construído a partir de modelos
Wavefront (`.obj`) com textura (somente `.obj`, sem arquivos `.mtl`). A cena é uma pequena ilha com:

* **Ambiente interno:** uma casa de madeira (quarto/oficina) com cama, baús, mesa de trabalho,
  fornalha, quadro, lanternas e vasos de flor.
* **Ambiente externo:** quintal com cercado e porco, lago (água, lava e pedra), árvore,
  "estação de trabalho" (mesa, fornalha e baú) e um pistão com alavanca.
* **Céu:** *SkyBox* simples: degradê de azul nos lados e embaixo, e **nuvens e sol só no teto** (face de cima). Sem bordas visíveis. A câmera fica restrita ao terreno e ao céu.

A câmera é livre (matrizes **Model x View x Projection**) e não há nenhum efeito de iluminação.

### Comandos

`W A S D - Move a câmera (frente / esquerda / trás / direita)`

`Mouse - Direciona a câmera`

`Scroll - Zoom (campo de visão)`

`R F - TRANSLAÇÃO: estende / recolhe o pistão`

`Q E - ROTAÇÃO: gira a cama 90° (anti-horário / horário)`

`Z X - ESCALA: o porco vira filhote / adulto`

`P - Ativa/Desativa a visualização poligonal (malhas)`

`Espaço - Reseta as condições iniciais`

`ESC - Sai`

### Onde cada requisito do enunciado é atendido

| Requisito | Onde |
|---|---|
| Interno + externo, com objetivo definido | lista `CENA` (campo `ambiente`) |
| 6+ modelos 3D com textura, de `.obj` diferentes | `objetos/modelos/*.obj` (18 modelos), carregados por `load_obj_and_texture` |
| 3+ modelos dentro e 3+ fora | `CENA`: dentro = cama, baú duplo, mesa, fornalha, quadro, lanterna, vaso; fora = porco, cerca, pistão, alavanca, árvore, lago... |
| Um item por `.obj` (sem cenas prontas) | `ferramentas/construir_modelos.py` separa a cena original em itens |
| Escala, rotação e translação em modelos diferentes, por teclado | porco (Z/X), cama (Q/E), pistão (R/F) em `matriz_model` |
| Pisos diferentes (dentro x fora) | piso de tábuas (`casa`) x grama (`terreno`) |
| Céu com textura (SkyBox) | `cria_skybox` + `tex/skybox/*.png` |
| Câmera restrita ao céu e ao terreno | `limita_camera` |
| `P` mostra/oculta a malha | `modo_wireframe` / `glPolygonMode` |
| Sem iluminação / só pipeline moderno | shaders `vertex_shader.vs` e `fragment_shader.fs` |

Integrantes:
```
Vinícius de Sá Ferreira -- 15491650
```

### Importando bibliotecas

* **PyOpenGL / glfw**: desenho 3D, janela e eventos de teclado/mouse.
* **numpy**: vértices e matrizes de transformação 4x4 (as matrizes de objeto são feitas à mão, como no Projeto 01).
* **PyGLM**: usado apenas na câmera (`lookAt`, `perspective`) e nos vetores da câmera.
* **PIL**: leitura das imagens de textura.

In [1]:
# Instala as dependências (executar apenas uma vez, se ainda não estiverem instaladas)
#!pip install PyOpenGL glfw PyGLM numpy pillow

In [2]:
import ctypes
import math
import os

import glfw                    # janela, contexto OpenGL e eventos
from OpenGL.GL import *        # API OpenGL (pipeline moderno)
import numpy as np             # vértices e matrizes
import glm                     # vetores/matrizes da câmera (lookAt, perspective)
from PIL import Image          # leitura das texturas

from shader_s import Shader    # classe auxiliar que compila/linka os shaders

### Inicializando a janela

A janela nasce escondida e só é exibida (`glfw.show_window`) logo antes do loop principal.

In [3]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

largura = 1000
altura = 700

window = glfw.create_window(largura, altura, "Projeto 02 - Casa Minecraft", None, None)

if window is None:
    glfw.terminate()
    raise RuntimeError("Failed to create GLFW window")

glfw.make_context_current(window)

### Constrói e compila os shaders

* `vertex_shader.vs`: aplica `projection * view * model` à posição e repassa a coordenada de textura.
* `fragment_shader.fs`: pinta o pixel com a cor da textura (multiplicada por uma "tinta" quando a
  textura é cinza, como a grama) e descarta os pixels transparentes (folhas, portas, flores).
  **Não há iluminação.**

In [4]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Carregando modelos (vértices e texturas)

`load_model_from_file` lê um arquivo **Wavefront** (`.obj`): vértices (`v`), coordenadas de textura (`vt`),
faces (`f`) e a textura de cada grupo de faces (`usemtl`). **Não usamos arquivos `.mtl`**: a própria linha
`usemtl` traz o caminho da imagem, relativo à pasta `tex/` e sem a extensão. Exemplo:

```
usemtl minecraft/block/oak_planks      ->      tex/minecraft/block/oak_planks.png
```

Assim um modelo pode ter **várias texturas** (por exemplo, a mesa de trabalho tem uma textura na frente,
outra nas laterais e outra em cima).

Cada modelo vira uma lista de **partes**: um intervalo contínuo de vértices (`inicio`, `qtd`) com a sua textura.
Todas as texturas são lidas uma única vez (`cache_texturas`) e ganham um id próprio com `glGenTextures`.


In [5]:
# Listas globais: os vértices e as coordenadas de textura de TODOS os modelos ficam aqui
# (vão para a GPU de uma vez só). Cada modelo guarda apenas onde os seus vértices começam.
vertices_list = []
textures_coord_list = []

# Cor multiplicada na textura quando ela é cinza (o Minecraft pinta grama/folhas/água por bioma)
TINTAS = {
    'grass_block_top': (0.569, 0.741, 0.349),
    'oak_leaves':      (0.467, 0.671, 0.184),
    'water_still':     (0.247, 0.463, 0.894),
    'water_flow':      (0.247, 0.463, 0.894),
}


def carrega_imagem(caminho):
    """Abre a imagem de textura. Se não existir, devolve um quadrado magenta e avisa."""
    if not os.path.exists(caminho):
        print('Textura nao encontrada:', caminho)
        return Image.new('RGBA', (16, 16), (255, 0, 255, 255))
    img = Image.open(caminho).convert('RGBA')
    w, h = img.size
    if h > w and h % w == 0:          # textura animada do Minecraft (tira vertical): usa só o 1o quadro
        img = img.crop((0, 0, w, w))
    return img


def eh_cinza(img):
    """True se a textura é toda em tons de cinza (precisa de tinta para ficar colorida)."""
    a = np.asarray(img).astype(int)
    pixels = a[..., :3][a[..., 3] > 0]
    if len(pixels) == 0:
        return False
    return np.abs(pixels[:, 0] - pixels[:, 1]).max() <= 3 and np.abs(pixels[:, 1] - pixels[:, 2]).max() <= 3


def load_texture_from_file(texture_id, img, wrap=GL_REPEAT):
    """Envia a imagem para a GPU no id de textura `texture_id`."""
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, wrap)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, wrap)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_NEAREST)  # visual "pixelado" do Minecraft
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_NEAREST)
    glPixelStorei(GL_UNPACK_ALIGNMENT, 1)
    image_data = img.tobytes("raw", "RGBA", 0, -1)     # -1: inverte o eixo vertical (origem da textura embaixo)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, img.size[0], img.size[1], 0, GL_RGBA, GL_UNSIGNED_BYTE, image_data)


PASTA_TEXTURAS = 'tex'   # as linhas `usemtl` dos .obj são caminhos relativos a esta pasta (sem extensão)

cache_texturas = {}   # (caminho, wrap) -> dados da textura (várias partes/modelos podem reaproveitar a mesma)


def carrega_textura(caminho, wrap=GL_REPEAT):
    """Carrega (uma única vez) a textura `caminho` e devolve {'id', 'tint', 'transparente'}."""
    chave = (os.path.normpath(caminho), wrap)
    if chave not in cache_texturas:
        img = carrega_imagem(caminho)
        nome = os.path.splitext(os.path.basename(caminho))[0]
        tint = TINTAS[nome] if (nome in TINTAS and eh_cinza(img)) else (1.0, 1.0, 1.0)
        texture_id = glGenTextures(1)
        load_texture_from_file(texture_id, img, wrap)
        cache_texturas[chave] = {'id': texture_id, 'tint': tint,
                                 'transparente': 'water' in nome}    # água é desenhada por último
    return cache_texturas[chave]


def load_model_from_file(filename):
    """Lê um Wavefront OBJ (vértices, coordenadas de textura, faces e a textura de cada grupo de faces)."""
    vertices = []
    texture_coords = []
    faces = []
    material = None

    for line in open(filename, "r"):                # para cada linha do arquivo .obj
        if line.startswith('#'):
            continue                                # ignora comentários
        values = line.split()                       # quebra a linha por espaço
        if not values:
            continue

        if values[0] == 'v':                        # vértice
            vertices.append([float(x) for x in values[1:4]])
        elif values[0] == 'vt':                     # coordenada de textura
            texture_coords.append([float(x) for x in values[1:3]])
        elif values[0] == 'usemtl':                 # as próximas faces usam esta textura (caminho em tex/, sem extensão)
            material = values[1]
        elif values[0] == 'f':                      # face: v/vt v/vt v/vt ...
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)
            faces.append((face, face_texture, material))

    return {'vertices': vertices, 'texture': texture_coords, 'faces': faces}


def circular_sliding_window_of_three(arr):
    """Triangula uma face com mais de 3 vértices (faces com 3 vértices passam direto)."""
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result


def load_obj_and_texture(arquivo_obj):
    """Importa um .obj com as suas texturas e devolve o modelo: {'nome', 'partes'}.
    Cada parte = faces de UMA textura (linha `usemtl`) = um intervalo de vértices + a textura dela."""
    modelo = load_model_from_file(arquivo_obj)

    faces_por_material = {}
    for face in modelo['faces']:
        faces_por_material.setdefault(face[2], []).append(face)

    partes = []
    for material, faces in faces_por_material.items():
        inicio = len(vertices_list)
        for face in faces:
            for vertice_id in circular_sliding_window_of_three(face[0]):
                vertices_list.append(modelo['vertices'][vertice_id - 1])
            for texture_id in circular_sliding_window_of_three(face[1]):
                textures_coord_list.append(modelo['texture'][texture_id - 1])
        if material is None:
            raise ValueError('%s: faces antes de qualquer linha usemtl (sem textura)' % arquivo_obj)
        tex = carrega_textura(os.path.join(PASTA_TEXTURAS, material + '.png'))
        partes.append({'inicio': inicio, 'qtd': len(vertices_list) - inicio,
                       'textura': tex['id'], 'tint': tex['tint'], 'transparente': tex['transparente']})

    return {'nome': os.path.splitext(os.path.basename(arquivo_obj))[0], 'partes': partes}

### Carregando todos os modelos da cena

Cada item da cena vem de **um `.obj` próprio** (`objetos/modelos/`), sem `.mtl`. Os vértices de cada modelo estão em
**coordenadas locais**: a origem é o pivô do item (o centro da sua base), então posicionar, girar ou escalar
um item é só montar a sua matriz **Model**.

Os modelos foram separados da cena original pela ferramenta `ferramentas/construir_modelos.py`
(que também corrigiu o porco, que estava inclinado e afundado no chão).

O **SkyBox** é um cubo com 6 texturas (`tex/skybox/`, geradas por `ferramentas/gerar_skybox.py`) montado por `cria_skybox`.
Os lados e a base são só um degradê de azul; as **nuvens e o sol ficam apenas na face de cima**.

In [6]:
PASTA_MODELOS = 'objetos/modelos'

NOMES_MODELOS = [
    # estrutura
    'terreno', 'casa', 'porta',
    # dentro da casa
    'cama', 'bau_duplo', 'quadro', 'lanterna', 'vaso_flor',
    # mesa de trabalho e fornalha aparecem dentro e fora
    'mesa_trabalho', 'fornalha',
    # fora da casa
    'porco', 'cerca', 'arvore', 'lago', 'bau', 'alavanca', 'piston_base', 'piston_braco',
]

modelos = {}
for nome in NOMES_MODELOS:
    modelos[nome] = load_obj_and_texture(os.path.join(PASTA_MODELOS, nome + '.obj'))
    print('%-14s %2d parte(s) com textura' % (nome, len(modelos[nome]['partes'])))


# ------------------------------------------------------------------ SkyBox
# Cubo [-1,1]^3 com uma textura por face. Para cada face: normal, eixo U e eixo V da textura.
# (A MESMA tabela é usada em ferramentas/gerar_skybox.py para gerar as imagens.)
FACES_SKYBOX = {
    'direita':  ((1, 0, 0),  (0, 0, -1), (0, 1, 0)),
    'esquerda': ((-1, 0, 0), (0, 0, 1),  (0, 1, 0)),
    'topo':     ((0, 1, 0),  (1, 0, 0),  (0, 0, -1)),
    'base':     ((0, -1, 0), (1, 0, 0),  (0, 0, 1)),
    'frente':   ((0, 0, 1),  (1, 0, 0),  (0, 1, 0)),
    'tras':     ((0, 0, -1), (-1, 0, 0), (0, 1, 0)),
}


def cria_skybox():
    """Monta o cubo do céu (6 faces x 2 triângulos) e carrega as 6 texturas."""
    partes = []
    for nome, (normal, eixo_u, eixo_v) in FACES_SKYBOX.items():
        n, U, V = (np.array(x, dtype=np.float32) for x in (normal, eixo_u, eixo_v))
        quad = [n + (2*u - 1)*U + (2*v - 1)*V for (u, v) in [(0, 0), (1, 0), (1, 1), (0, 1)]]
        uvs = [(0, 0), (1, 0), (1, 1), (0, 1)]
        inicio = len(vertices_list)
        for i in (0, 1, 2, 0, 2, 3):                       # 2 triângulos por face
            vertices_list.append(quad[i].tolist())
            textures_coord_list.append(list(uvs[i]))
        # CLAMP_TO_EDGE: evita que a textura "vaze" para o outro lado e crie linhas nas emendas
        tex = carrega_textura(os.path.join('tex', 'skybox', nome + '.png'), wrap=GL_CLAMP_TO_EDGE)
        partes.append({'inicio': inicio, 'qtd': 6, 'textura': tex['id'],
                       'tint': (1.0, 1.0, 1.0), 'transparente': False})
    return {'nome': 'skybox', 'partes': partes}


skybox = cria_skybox()

terreno         5 parte(s) com textura
casa            3 parte(s) com textura
porta           2 parte(s) com textura
cama            1 parte(s) com textura
bau_duplo       2 parte(s) com textura
quadro          1 parte(s) com textura
lanterna        1 parte(s) com textura
vaso_flor       3 parte(s) com textura
mesa_trabalho   4 parte(s) com textura
fornalha        3 parte(s) com textura
porco           1 parte(s) com textura
cerca           1 parte(s) com textura
arvore          3 parte(s) com textura
lago            4 parte(s) com textura
bau             1 parte(s) com textura
alavanca        2 parte(s) com textura
piston_base     2 parte(s) com textura
piston_braco    2 parte(s) com textura


### A cena: instâncias dos modelos

Cada elemento de `CENA` é um `Objeto`: **qual modelo usar** e **onde/como colocá-lo** (posição, rotação em Y e
escala). O mesmo modelo pode aparecer várias vezes (2 lanternas, 2 vasos, 2 baús duplos, 2 mesas, 2 fornalhas).

Sistema de coordenadas do mundo: `y = 0` é o chão (topo da grama e do piso de tábuas); `+z` aponta para o fundo da
casa; a unidade é o "bloco" do Minecraft (1 bloco ≈ 1 metro). O quintal fica em `z < 0` e a casa em `z > 0`.

Os objetos animados pelo teclado são identificados pelo `id`: `cama` (rotação), `porco` (escala) e
`piston_braco` (translação).

In [7]:
class Objeto:
    """Uma instância de um modelo na cena.

    Atributos:
        id: nome único do objeto (usado para aplicar o comportamento específico de cada um).
        modelo: nome do modelo (chave de `modelos`) que será desenhado.
        pos: posição (x, y, z) do pivô do modelo no mundo.
        rot_y: rotação (graus) em torno do eixo Y.
        escala: fator de escala (sx, sy, sz).
        ambiente: 'interno', 'externo' ou 'estrutura' (terreno/casa, que delimitam os ambientes).
    """
    def __init__(self, id, modelo, pos, rot_y=0.0, escala=(1.0, 1.0, 1.0), ambiente='externo'):
        self.id = id
        self.modelo = modelo
        self.pos = pos
        self.rot_y = rot_y
        self.escala = escala
        self.ambiente = ambiente


CENA = [
    # ---------------- estrutura: piso externo (grama), casa (piso interno de tábuas) e porta
    Objeto('terreno',   'terreno', (0, 0, 0),   ambiente='estrutura'),
    Objeto('casa',      'casa',    (0, 0, 0),   ambiente='estrutura'),
    Objeto('porta',     'porta',   (3, 0, 1),   ambiente='estrutura'),

    # ---------------- ambiente INTERNO (dentro da casa)
    Objeto('cama',        'cama',          (3, 0, 5.5),   ambiente='interno'),             # ROTAÇÃO (Q/E)
    Objeto('bau_duplo_1', 'bau_duplo',     (0.5, 0, 5.5), ambiente='interno'),
    Objeto('bau_duplo_2', 'bau_duplo',     (5.5, 0, 5.5), rot_y=180, ambiente='interno'),  # mesmo modelo, virado p/ o centro
    Objeto('mesa_int',    'mesa_trabalho', (0.5, 0, 4),   ambiente='interno'),
    Objeto('fornalha_int','fornalha',      (5.5, 0, 4),   rot_y=-90, ambiente='interno'),  # frente virada p/ o centro da sala
    # quadro 1x1 com textura 2:1  ->  escala 2 em X; centrado sobre a janela e a cama
    Objeto('quadro',      'quadro',        (3, 2, 6.47),  escala=(2, 1, 1), ambiente='interno'),
    Objeto('lanterna_1',  'lanterna',      (2.5, 3.062, 5), ambiente='interno'),
    Objeto('lanterna_2',  'lanterna',      (3.5, 3.062, 5), ambiente='interno'),
    Objeto('vaso_1',      'vaso_flor',     (2.5, 1, 7),   ambiente='interno'),
    Objeto('vaso_2',      'vaso_flor',     (3.5, 1, 7),   ambiente='interno'),

    # ---------------- ambiente EXTERNO (quintal)
    Objeto('cerca',        'cerca',        (5.5, 0, -3)),
    Objeto('porco',        'porco',        (5.5, 0, -3), rot_y=60),                        # ESCALA (Z/X)
    Objeto('arvore',       'arvore',       (-4.5, 0, -4)),
    Objeto('lago',         'lago',         (0, 0, -4)),
    Objeto('piston_base',  'piston_base',  (-0.5, 0, -6)),
    Objeto('piston_braco', 'piston_braco', (-0.5, 0, -6)),                                 # TRANSLAÇÃO (R/F)
    Objeto('alavanca',     'alavanca',     (0.5, 0, -6)),
    Objeto('mesa_ext',     'mesa_trabalho', (4.5, 0, -6)),
    Objeto('fornalha_ext', 'fornalha',     (3.5, 0, -6)),
    Objeto('bau',          'bau',          (5.5, 0, -6)),
]

### Enviando vértices e coordenadas de textura para a GPU (dois buffers)

Um VBO para as posições e outro para as coordenadas de textura (todos os modelos e o céu juntos).

In [8]:
buffer_VBO = glGenBuffers(2)

# ---- posições
posicoes = np.array(vertices_list, dtype=np.float32)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, posicoes.nbytes, posicoes, GL_STATIC_DRAW)
stride = posicoes.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

# ---- coordenadas de textura
coords_textura = np.array(textures_coord_list, dtype=np.float32)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, coords_textura.nbytes, coords_textura, GL_STATIC_DRAW)
stride = coords_textura.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")
glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)

# ---- uniforms (matrizes Model/View/Projection, tinta e unidade de textura)
loc_model = glGetUniformLocation(program, "model")
loc_view = glGetUniformLocation(program, "view")
loc_projection = glGetUniformLocation(program, "projection")
loc_tint = glGetUniformLocation(program, "tint")
glUniform1i(glGetUniformLocation(program, "samplerTexture"), 0)   # unidade de textura 0

### Matrizes de transformação 3D (Model)

As mesmas funções do Projeto 01: matrizes 4x4 "achatadas" (linha a linha) enviadas ao shader com
`transpose=GL_TRUE`. A matriz **Model** de cada objeto é `T * R * S`: primeiro escala, depois rotação
(em torno do pivô do modelo) e por fim a translação para a posição no mundo.

In [9]:
def multiplica_matriz(a, b):
    """Multiplica duas matrizes 4x4 (a * b), no formato achatado (1,16)."""
    m_a = a.reshape(4, 4)
    m_b = b.reshape(4, 4)
    m_c = np.dot(m_a, m_b)
    c = m_c.reshape(1, 16)
    return c


def translacao(tx, ty, tz):
    """Matriz de translação: desloca um ponto em (tx, ty, tz)."""
    return np.array([
        1, 0, 0, tx,
        0, 1, 0, ty,
        0, 0, 1, tz,
        0, 0, 0,  1
    ], np.float32)


def rotacao(rx, ry, rz):
    """Matriz de rotação combinada (Z * Y * X), com os ângulos (em radianos) em torno de cada eixo."""
    c, s = np.cos(rx), np.sin(rx)
    X = np.array([
        1, 0,  0, 0,
        0, c, -s, 0,
        0, s,  c, 0,
        0, 0,  0, 1
    ], np.float32)

    c, s = np.cos(ry), np.sin(ry)
    Y = np.array([
         c, 0, s, 0,
         0, 1, 0, 0,
        -s, 0, c, 0,
         0, 0, 0, 1
    ], np.float32)

    c, s = np.cos(rz), np.sin(rz)
    Z = np.array([
        c, -s, 0, 0,
        s,  c, 0, 0,
        0,  0, 1, 0,
        0,  0, 0, 1
    ], np.float32)

    YX = multiplica_matriz(Y, X)
    return multiplica_matriz(Z, YX)


def escala(sx, sy, sz):
    """Matriz de escala não-uniforme, com fatores sx, sy e sz em cada eixo."""
    return np.array([
        sx,  0,  0, 0,
         0, sy,  0, 0,
         0,  0, sz, 0,
         0,  0,  0, 1
    ], np.float32)

### Estado da cena e controles

* **Pistão (translação, R/F):** a cabeça e a haste do pistão (`piston_braco`) sobem/descem em Y em relação à base.
  `ext_piston` vai de ~0 (recolhido) a 1 (estendido = 1 bloco). A cena começa com o pistão estendido, como no mundo original.
* **Cama (rotação, Q/E):** cada toque soma/subtrai 90° ao ângulo-alvo (`ang_cama_alvo`) e a cama gira até lá de forma animada,
  em torno do próprio centro.
* **Porco (escala, Z/X):** `esc_porco` vai de 0.5 (filhote) a 1.0 (adulto). A escala é feita a partir dos pés (origem do modelo),
  então o porco continua no chão.

As teclas pressionadas ficam no conjunto `teclas`; a cada quadro, `atualiza` usa o tempo decorrido (`deltaTime`)
para que a velocidade seja a mesma em qualquer computador.

In [10]:
# ------------------------------------------------------------------ parâmetros das animações
EXT_MIN, EXT_MAX = 0.004, 1.0     # curso do pistão (em blocos). EXT_MIN > 0 evita z-fighting com a base
VEL_PISTON = 1.0                  # blocos por segundo
VEL_CAMA = 180.0                  # graus por segundo
ESC_FILHOTE, ESC_ADULTO = 0.5, 1.0
VEL_ESCALA = 0.5                  # variação da escala por segundo

# ------------------------------------------------------------------ estado atual (valores iniciais)
ext_piston = EXT_MAX              # TRANSLAÇÃO: pistão estendido
ang_cama = 0.0                    # ROTAÇÃO: ângulo atual da cama (graus)
ang_cama_alvo = 0.0               #          ângulo para o qual a cama está girando (múltiplo de 90)
esc_porco = ESC_ADULTO            # ESCALA: porco adulto

modo_wireframe = False            # P: malha poligonal
teclas = set()                    # teclas atualmente pressionadas

### Câmera (matriz **View**) e limites

* `W A S D` movem a câmera; o mouse direciona; o scroll muda o campo de visão.
* `limita_camera` mantém a câmera **dentro do terreno** (em X e Z), **acima do piso** e **abaixo do teto do céu**.
  Ela pode atravessar os modelos livremente (a casa, os móveis...), mas não o piso nem o céu.
* O **SkyBox** é desenhado sempre centrado na câmera, então o céu fica "no infinito" e suas bordas nunca aparecem.

In [11]:
# ------------------------------------------------------------------ limites do cenário
# Caixa do terreno (em x e z), calculada a partir da malha do modelo 'terreno'
_p = posicoes[modelos['terreno']['partes'][0]['inicio']:
              modelos['terreno']['partes'][-1]['inicio'] + modelos['terreno']['partes'][-1]['qtd']]
_t = np.array([[o.pos[0], o.pos[2]] for o in CENA if o.modelo == 'terreno'][0])
MARGEM = 0.3                                   # a câmera não encosta exatamente na borda
LIM_X = (_p[:, 0].min() + _t[0] + MARGEM, _p[:, 0].max() + _t[0] - MARGEM)
LIM_Z = (_p[:, 2].min() + _t[1] + MARGEM, _p[:, 2].max() + _t[1] - MARGEM)
ALTURA_MIN = 0.4                               # acima do piso (y = 0)
ALTURA_MAX = 40.0                              # "teto" do céu
print('limites da camera: x', LIM_X, 'z', LIM_Z, 'y', (ALTURA_MIN, ALTURA_MAX))

TAMANHO_CEU = 100.0                            # lado/2 do cubo do céu (menor que o plano far = 300)

# ------------------------------------------------------------------ câmera
POS_INICIAL = glm.vec3(3.0, 6.0, -8.0)         # no quintal, olhando para a casa
YAW_INICIAL, PITCH_INICIAL = 90.0, -24.0       # yaw 90 = olhando para +z
VEL_CAMERA = 6.0                               # blocos por segundo

cameraPos = glm.vec3(POS_INICIAL)
cameraUp = glm.vec3(0.0, 1.0, 0.0)
yaw = YAW_INICIAL
pitch = PITCH_INICIAL


def calcula_front(yaw, pitch):
    """Direção para onde a câmera olha, a partir dos ângulos yaw/pitch (em graus)."""
    front = glm.vec3(glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch)),
                     glm.sin(glm.radians(pitch)),
                     glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch)))
    return glm.normalize(front)


cameraFront = calcula_front(yaw, pitch)

firstMouse = True
lastX = largura / 2.0
lastY = altura / 2.0
fov = 60.0

# tempo
deltaTime = 0.0


def limita_camera():
    """Mantém a câmera dentro do terreno e do céu (os modelos podem ser atravessados)."""
    cameraPos.x = min(max(cameraPos.x, LIM_X[0]), LIM_X[1])
    cameraPos.z = min(max(cameraPos.z, LIM_Z[0]), LIM_Z[1])
    cameraPos.y = min(max(cameraPos.y, ALTURA_MIN), ALTURA_MAX)


def reseta_cena():
    """Volta tudo para as condições iniciais (Espaço)."""
    global ext_piston, ang_cama, ang_cama_alvo, esc_porco, modo_wireframe
    global cameraPos, yaw, pitch, cameraFront, fov
    ext_piston = EXT_MAX
    ang_cama = ang_cama_alvo = 0.0
    esc_porco = ESC_ADULTO
    modo_wireframe = False
    cameraPos = glm.vec3(POS_INICIAL)
    yaw, pitch, fov = YAW_INICIAL, PITCH_INICIAL, 60.0
    cameraFront = calcula_front(yaw, pitch)


def key_event(window, key, scancode, action, mods):
    """Callback de teclado do GLFW: guarda as teclas pressionadas e trata os comandos de um toque só."""
    global modo_wireframe, ang_cama_alvo

    if action == glfw.PRESS:
        teclas.add(key)

        if key == glfw.KEY_ESCAPE:
            glfw.set_window_should_close(window, True)

        # P: alterna entre sólido e malha poligonal
        if key == glfw.KEY_P:
            modo_wireframe = not modo_wireframe

        # Q/E: ROTAÇÃO da cama, de 90 em 90 graus
        if key == glfw.KEY_Q:
            ang_cama_alvo += 90.0
        if key == glfw.KEY_E:
            ang_cama_alvo -= 90.0

        # Espaço: reseta tudo
        if key == glfw.KEY_SPACE:
            reseta_cena()

    elif action == glfw.RELEASE:
        teclas.discard(key)


def framebuffer_size_callback(window, w, h):
    global largura, altura
    largura, altura = max(w, 1), max(h, 1)
    glViewport(0, 0, w, h)


def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch

    if firstMouse:
        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos          # invertido: as coordenadas y da janela crescem para baixo
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1
    yaw += xoffset * sensitivity
    pitch += yoffset * sensitivity
    pitch = min(max(pitch, -89.0), 89.0)

    cameraFront = calcula_front(yaw, pitch)


def scroll_callback(window, xoffset, yoffset):
    global fov
    fov = min(max(fov - yoffset, 1.0), 90.0)


glfw.set_key_callback(window, key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

limites da camera: x (np.float64(-7.7), np.float64(7.7)) z (np.float64(-8.2), np.float64(8.2)) y (0.4, 40.0)


### Matrizes Model, View e Projection

* **model**: uma por objeto, `T * R * S`. É aqui que o porco muda de escala, a cama gira e o pistão se desloca.
* **view**: `glm.lookAt` a partir da posição/direção da câmera.
* **projection**: perspectiva.

In [12]:
def matriz_model(obj):
    """Matriz Model do objeto: T(pos) * Ry(rot) * S(escala), com os efeitos controlados pelo teclado."""
    pos = list(obj.pos)
    rot_y = obj.rot_y
    esc = obj.escala

    if obj.id == 'cama':
        # ROTAÇÃO (Q/E): gira em torno do pivô do modelo (centro da base da cama)
        rot_y += ang_cama
    elif obj.id == 'porco':
        # ESCALA (Z/X): o pivô está nos pés, então ao crescer/encolher o porco continua no chão
        esc = (esc_porco, esc_porco, esc_porco)
    elif obj.id == 'piston_braco':
        # TRANSLAÇÃO (R/F): cabeça + haste deslizam em Y (o modelo está na posição "estendido")
        pos[1] += ext_piston - EXT_MAX

    m = escala(*esc)
    m = multiplica_matriz(rotacao(0, np.radians(rot_y), 0), m)
    m = multiplica_matriz(translacao(*pos), m)
    return m


def view():
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp)
    return np.array(mat_view)


def projection():
    # parâmetros: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura / altura, 0.1, 300.0)
    return np.array(mat_projection)

### Atualização e desenho da cena

* `atualiza(dt)`: move a câmera e faz avançar as animações de acordo com as teclas pressionadas.
* `desenha_cena()`: primeiro o **céu** (sempre preenchido, sem escrever profundidade), depois os objetos
  (peças opacas e, por último, as transparentes - a água).

In [13]:
def atualiza(dt):
    """Avança a câmera e as animações em `dt` segundos."""
    global cameraPos, ext_piston, ang_cama, esc_porco

    # --- câmera (WASD)
    v = VEL_CAMERA * dt
    direita = glm.normalize(glm.cross(cameraFront, cameraUp))
    if glfw.KEY_W in teclas: cameraPos += v * cameraFront
    if glfw.KEY_S in teclas: cameraPos -= v * cameraFront
    if glfw.KEY_A in teclas: cameraPos -= v * direita
    if glfw.KEY_D in teclas: cameraPos += v * direita
    limita_camera()

    # --- TRANSLAÇÃO: pistão (R estende, F recolhe)
    if glfw.KEY_R in teclas: ext_piston += VEL_PISTON * dt
    if glfw.KEY_F in teclas: ext_piston -= VEL_PISTON * dt
    ext_piston = min(max(ext_piston, EXT_MIN), EXT_MAX)

    # --- ROTAÇÃO: cama gira até o ângulo-alvo (múltiplo de 90 graus)
    passo = VEL_CAMA * dt
    ang_cama += min(max(ang_cama_alvo - ang_cama, -passo), passo)

    # --- ESCALA: porco (Z vira filhote, X vira adulto)
    if glfw.KEY_Z in teclas: esc_porco -= VEL_ESCALA * dt
    if glfw.KEY_X in teclas: esc_porco += VEL_ESCALA * dt
    esc_porco = min(max(esc_porco, ESC_FILHOTE), ESC_ADULTO)


def desenha_partes(modelo, transparente=None):
    """Desenha as partes (textura + intervalo de vértices) de um modelo.
    transparente=None desenha todas; True/False desenha só as transparentes/opacas."""
    for parte in modelo['partes']:
        if transparente is not None and parte['transparente'] != transparente:
            continue
        glBindTexture(GL_TEXTURE_2D, parte['textura'])
        glUniform3f(loc_tint, *parte['tint'])
        glDrawArrays(GL_TRIANGLES, parte['inicio'], parte['qtd'])


def desenha_cena():
    # ---- 1. céu: cubo centrado na câmera (nunca fica "perto" nem mostra bordas), sempre preenchido
    glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)
    glDepthMask(GL_FALSE)                                  # o céu não escreve no z-buffer
    mat_ceu = multiplica_matriz(translacao(cameraPos.x, cameraPos.y, cameraPos.z),
                                escala(TAMANHO_CEU, TAMANHO_CEU, TAMANHO_CEU))
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_ceu)
    desenha_partes(skybox)
    glDepthMask(GL_TRUE)

    # ---- 2. objetos da cena (P liga/desliga a malha poligonal)
    glPolygonMode(GL_FRONT_AND_BACK, GL_LINE if modo_wireframe else GL_FILL)
    matrizes = [matriz_model(obj) for obj in CENA]
    for transparente in (False, True):                     # opacos primeiro, água por último
        for obj, mat_model in zip(CENA, matrizes):
            glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            desenha_partes(modelos[obj.modelo], transparente)

### Nesse momento, nós exibimos a janela!

In [14]:
glfw.show_window(window)

### Loop principal da janela

In [15]:
glEnable(GL_DEPTH_TEST)    # importante para 3D
glEnable(GL_BLEND)         # transparência (água)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)

ultimoFrame = glfw.get_time()
ultimoTitulo = 0.0

while not glfw.window_should_close(window):

    agora = glfw.get_time()
    deltaTime = min(agora - ultimoFrame, 0.1)    # limita o passo (ex.: janela arrastada)
    ultimoFrame = agora

    glfw.poll_events()
    atualiza(deltaTime)

    glClearColor(0.53, 0.81, 0.92, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glUniformMatrix4fv(loc_view, 1, GL_TRUE, view())
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, projection())

    desenha_cena()

    # mostra o estado dos objetos animados no título da janela (atualiza 5x por segundo)
    if agora - ultimoTitulo > 0.2:
        ultimoTitulo = agora
        glfw.set_window_title(window, "Projeto 02 | pistao %3d%% | cama %4d graus | porco %.2fx | malha: %s" % (
            round(100 * (ext_piston - EXT_MIN) / (EXT_MAX - EXT_MIN)), round(ang_cama) % 360,
            esc_porco, "ligada" if modo_wireframe else "desligada"))

    glfw.swap_buffers(window)

glfw.terminate()